In [ ]:
"""ACCESS-C demo/test."""

%load_ext autoreload
%autoreload 2
import glob
import shutil
import yaml
import numpy as np
import networkx as nx
import xarray as xr
from pathlib import Path

import thuner.data as data
import thuner.track.track as track
import thuner.option as option
import thuner.analyze as analyze
import thuner.parallel as parallel
import thuner.visualize as visualize
import thuner.attribute as attribute
import thuner.default as default
import thuner.config as config
import thuner.utils as utils
import thuner.match as match
import thuner.grid as grid

In [ ]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = config.get_outputs_directory()
output_parent = base_local / f"runs/access_c/access_c_demo"
options_directory = output_parent / "options"
visualize_directory = output_parent / "visualize"

In [ ]:
# Delete the output directory for the run if it already exists
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)

In [ ]:
# Create the dataset options
# For model datasets we generally need to specify which model run we want, in 
# addition to the start and end times. Typically we want to discard spin up times.
run_start = "2021-12-01T12:00:00" # The start time of the run we want
start = "2021-12-02T06:00:00" # The start time of the data we want to analyze. 
end = "2021-12-02T12:00:00" # The end time of the data we want to analyze.
times_dict = {"start": start, "end": end, "run_start": run_start}

access_1km_options = data.access.ACCESSOptions(
    **times_dict, name="access_1km", filename="radar_refl_1km.nc"
)
access_max_col_options = data.access.ACCESSOptions(
    **times_dict, name="access_maxcol", filename="maxcol_refl.nc"
)

In [ ]:
datasets=[access_1km_options, access_max_col_options]
data_options = option.data.DataOptions(datasets=datasets)
data_options.to_yaml(options_directory / "data.yml")

grid_options = option.grid.GridOptions()
grid_options.to_yaml(options_directory / "grid.yml")

track_options = default.access_c_track()
track_options.to_yaml(options_directory / "track.yml")

In [ ]:
times = utils.generate_dataset_times(data_options.dataset_by_name("access_1km"))
args = [times, data_options, grid_options, track_options]
parallel.track(
    *args, output_directory=output_parent, dataset_name="access_1km", num_processes=3
)
# track.track(*args, output_directory=output_parent)

In [ ]:
analysis_options = analyze.mcs.AnalysisOptions()
analysis_options.to_yaml(options_directory / "analysis.yml")
analyze.mcs.process_velocities(output_parent, profile_dataset=None)
analyze.mcs.quality_control(output_parent, analysis_options)

In [ ]:
style = "presentation"
attribute_handlers = default.grouped_attribute_handlers(output_parent, style)
kwargs = {"name": "mcs_attributes", "object_name": "mcs", "style": style}
kwargs.update({"attribute_handlers": attribute_handlers})
figure_options = option.visualize.GroupedHorizontalAttributeOptions(**kwargs)
args = [output_parent, start, end, figure_options, "access_1km"]
args_dict = {"parallel_figure": True, "by_date": False, "num_processes": 4}
visualize.attribute.series(*args, **args_dict)

In [ ]:
dt = xr.open_datatree(output_parent / "output.zarr", engine="zarr")

In [ ]:
quality = dt.analysis.quality.ds

In [ ]:
quality_df = quality.to_dataframe().set_index(quality.index_columns)
raw_sample = quality_df.where(quality_df["duration"]).dropna() 
mcs_count = len(raw_sample.index.get_level_values("universal_id").unique())
print(mcs_count)

In [ ]:
velocity = dt.analysis.velocities.ds
velocity_df = velocity.to_dataframe().set_index(velocity.index_columns)
raw_sample = velocity_df.where(quality_df["duration"]).dropna()

In [ ]:
average_velocities = raw_sample.reset_index()[["u", "v"]].mean(axis=0)
print(average_velocities)